In [1]:
# Imports

from __future__ import annotations

import json
import math
import os
import random
import sys
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights, ResNet50_Weights

from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    auc as sk_auc,
    silhouette_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from pytorch_grad_cam.utils.image import show_cam_on_image

In [2]:
# ==============================
# Configuração global
# ==============================

SEED = 42
TASK = "Ischaemia"
DATA_ROOT = Path("../data/ischaemia")
MODEL_NAME = "efficientnet"   # "efficientnet" ou "resnet50"
IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
PATIENCE = 8
N_FOLDS = 5
VAL_N_SPLITS = 4  # usado para criar a validação dentro do conjunto de treino+val
PCA_N_COMPONENTS = 50
NUM_WORKERS = 0
SAVE_GRADCAM_EXAMPLES = True
MAX_GRADCAM_IMAGES_PER_FOLD = 4
USE_CODECARBON = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT_OUTPUT = Path("../reports")
DIR_MODELS = Path("../models") / TASK.lower()
DIR_METRICS = ROOT_OUTPUT / "metrics"
DIR_FIGURES = ROOT_OUTPUT / "figures"
DIR_PCA_FIG = DIR_FIGURES / "pca"
DIR_GRADCAM = DIR_FIGURES / "gradcam"
DIR_EMISSIONS = ROOT_OUTPUT / "emissions"

for directory in [DIR_MODELS, DIR_METRICS, DIR_FIGURES, DIR_PCA_FIG, DIR_GRADCAM, DIR_EMISSIONS]:
    directory.mkdir(parents=True, exist_ok=True)

FOLD_COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
CLASS_COLORS = {0: "#3B8BD4", 1: "#E8593C"}
SPLIT_COLORS = {"train": "#4C72B0", "val": "#55A868", "test": "#DD8452"}

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def set_seed(seed: int = SEED) -> None:
    """Garante reprodutibilidade razoável."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Essas flags deixam o treino mais determinístico.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ==============================
# Imports opcionais
# ==============================

try:
    import codecarbon
except Exception:  # pragma: no cover
    codecarbon = None

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
except Exception:  # pragma: no cover
    GradCAM = None
    show_cam_on_image = None


def import_ocpc_multiclass():
    """
    Importa o OCPC de forma tolerante a falhas.

    O pacote publicado pode quebrar em alguns ambientes por causa de imports
    absolutos internos (por exemplo, `from utils import *`).
    Este helper tenta contornar isso adicionando o diretório do pacote ao sys.path.
    """
    try:
        from ocpc_py import MultiClassPC  # type: ignore
        return MultiClassPC
    except Exception:
        pass

    try:
        import ocpc_py  # type: ignore

        pkg_dir = os.path.dirname(ocpc_py.__file__)
        if pkg_dir not in sys.path:
            sys.path.append(pkg_dir)

        from ocpc_py.Models import MultiClassPC  # type: ignore
        return MultiClassPC
    except Exception as exc:  # pragma: no cover
        raise ImportError(
            "Não foi possível importar `MultiClassPC` de `ocpc_py`. "
            "Verifique se `ocpc-py` está instalado no mesmo ambiente do notebook."
        ) from exc


MultiClassPC = import_ocpc_multiclass()

In [ ]:
# ==============================
# Patch ocpc_py – compatibilidade Python 3.14 + NumPy moderno
# ==============================
import importlib, os, sys, re, ast, warnings

def _patch_ocpc_utils() -> bool:
    """
    Corrige ocpc_py/utils.py in-place para Python 3.14 + NumPy moderno.
    Retorna True se o patch foi aplicado (ou já estava aplicado), False se falhou.
    """
    try:
        import ocpc_py
    except ImportError:
        print("[patch] ERRO: ocpc_py não está instalado.")
        return False

    utils_path = os.path.join(os.path.dirname(ocpc_py.__file__), "utils.py")
    print(f"[patch] Arquivo alvo: {utils_path}")

    if not os.path.isfile(utils_path):
        print("[patch] ERRO: utils.py não encontrado no pacote.")
        return False

    with open(utils_path, "r", encoding="utf-8") as fh:
        src = fh.read()

    print(f"[patch] Arquivo lido ({len(src)} chars). Procurando padrões...")

    # -- Mostra as linhas problemáticas para diagnóstico --
    for i, line in enumerate(src.splitlines(), 1):
        if "lengths[vr" in line or "rest[i,:,vr" in line:
            print(f"  Linha {i}: {line.rstrip()}")

    patched = src
    applied = []

    # Estratégia: regex tolerante a espaços ao redor de colchetes
    fixes = [
        # Fix 1: y[i,0] = y[i,0] + lengths[vr[i]]
        (
            r"(y\[i\s*,\s*0\]\s*=\s*y\[i\s*,\s*0\]\s*\+\s*)lengths\[vr\[i\]\]",
            r"\1float(lengths[int(vr[i])])",
            "Fix1: lengths[vr[i]] -> float(lengths[int(vr[i])])"
        ),
        # Fix 2: y[i,:] = rest[i,:,vr[i]]
        (
            r"(y\[i\s*,\s*:\]\s*=\s*)rest\[i\s*,\s*:\s*,\s*vr\[i\]\]",
            r"\1rest[i,:,int(vr[i])]",
            "Fix2: rest[...vr[i]] -> rest[...int(vr[i])]"
        ),
    ]

    for pattern, replacement, label in fixes:
        new_src = re.sub(pattern, replacement, patched)
        if new_src != patched:
            patched = new_src
            applied.append(label)
            print(f"  [OK] {label}")
        else:
            # Verifica se já foi corrigido anteriormente
            if "int(vr[i])" in patched:
                print(f"  [SKIP] {label} – já estava corrigido.")
                applied.append(label)  # conta como OK
            else:
                print(f"  [WARN] Padrão não encontrado: {label}")

    if patched != src:
        with open(utils_path, "w", encoding="utf-8") as fh:
            fh.write(patched)
        print(f"[patch] utils.py gravado com {len(applied)} correção(ões).")
    else:
        print("[patch] Nenhuma alteração necessária (já corrigido ou padrão diferente).")

    return len(applied) > 0


def _reload_ocpc() -> type:
    """
    Remove ocpc_py do cache e reimporta do zero.
    Mais confiável que importlib.reload() para módulos com dependências internas.
    """
    mods_to_remove = [k for k in sys.modules if k.startswith("ocpc_py")]
    for mod in mods_to_remove:
        del sys.modules[mod]
    print(f"[patch] {len(mods_to_remove)} módulos ocpc_py removidos do cache.")

    from ocpc_py import MultiClassPC  # type: ignore
    print("[patch] MultiClassPC reimportado com sucesso.")
    return MultiClassPC


def _smoke_test(MultiClassPC) -> None:
    """
    Teste rápido com dados sintéticos para confirmar que o patch funcionou
    ANTES de rodar o pipeline completo.
    """
    import numpy as np
    print("[patch] Executando smoke test...")
    rng = np.random.default_rng(42)
    # Dados mínimos: 2 classes, 5 features, 40 amostras
    X = rng.standard_normal((40, 5)).astype(np.float64)
    y = np.array([0]*20 + [1]*20, dtype=np.int32)
    try:
        clf = MultiClassPC()
        clf.fit(X, y)
        preds = clf.predict(X[:5])
        print(f"[patch] Smoke test OK – predições de amostra: {list(preds[:5])}")
    except Exception as exc:
        print(f"[patch] SMOKE TEST FALHOU: {exc}")
        print("[patch] O pipeline provavelmente vai falhar no OCPC.")
        print("[patch] Sugestão: abra o arquivo abaixo e corrija manualmente a linha indicada.")
        import ocpc_py
        utils_path = os.path.join(os.path.dirname(ocpc_py.__file__), "utils.py")
        print(f"         Arquivo: {utils_path}")
        print("         Procure por: lengths[vr[i]]")
        print("         Substitua por: float(lengths[int(vr[i])])")
        raise RuntimeError("Patch do ocpc_py falhou – veja instruções acima.") from exc


# ── Execução ──
_patch_ocpc_utils()
MultiClassPC = _reload_ocpc()
_smoke_test(MultiClassPC)
print("[patch] Tudo pronto. Pode executar o pipeline.")


In [3]:
# ==============================
# Dataset e transformações
# ==============================

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()),
])


class DFUDataset(Dataset):
    """
    Dataset de lesões no pé diabético.

    Estrutura esperada:
        root_dir/
            Aug-Positive/
            Aug-Negative/

    O identificador do paciente é extraído do prefixo do nome do arquivo.
    Ex.: pat042_1X_M.jpg -> paciente pat042

    Esse identificador é usado nos splits com grupo para evitar leakage
    entre imagens do mesmo paciente.
    """

    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths: List[Path] = []
        self.labels: List[int] = []
        self.identifiers: List[str] = []

        positive_dir = self.root_dir / "Aug-Positive"
        negative_dir = self.root_dir / "Aug-Negative"

        if not positive_dir.exists() or not negative_dir.exists():
            raise FileNotFoundError(
                f"Estrutura de pastas não encontrada em {self.root_dir}. "
                "Esperado: Aug-Positive e Aug-Negative."
            )

        for class_dir, label in [(positive_dir, 1), (negative_dir, 0)]:
            for image_path in sorted(class_dir.iterdir()):
                if image_path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
                    continue
                patient_id = image_path.stem.split("_")[0]
                self.image_paths.append(image_path)
                self.labels.append(label)
                self.identifiers.append(patient_id)

        if len(self.image_paths) == 0:
            raise RuntimeError(f"Nenhuma imagem encontrada em {self.root_dir}")

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int):
        image_path = self.image_paths[index]
        image = Image.open(image_path).convert("RGB")
        label = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)

        # Mantemos o caminho para facilitar Grad-CAM e auditoria do fold.
        return image, label, str(image_path)

# ==============================
# Utilidades de split
# ==============================

def build_subset(dataset: DFUDataset, indices: Sequence[int], transform) -> Dataset:
    """
    Cria um subset com transform específico.

    Em vez de mudar o transform do dataset original no meio do pipeline,
    clonamos uma visão do dataset com o transform apropriado.
    """
    subset_dataset = DFUDataset(dataset.root_dir, transform=transform)
    return Subset(subset_dataset, list(indices))


def get_nested_group_split(
    labels: Sequence[int],
    groups: Sequence[str],
    n_outer_splits: int = N_FOLDS,
    n_inner_splits: int = VAL_N_SPLITS,
    seed: int = SEED,
):
    """
    Gera splits aninhados:
    - outer: train_val vs test
    - inner: train vs val, apenas dentro do conjunto train_val

    Ambos respeitam estratificação aproximada por rótulo e separação por grupos.
    """
    labels = np.asarray(labels)
    groups = np.asarray(groups)

    outer_cv = StratifiedGroupKFold(n_splits=n_outer_splits, shuffle=True, random_state=seed)

    for outer_fold, (train_val_idx, test_idx) in enumerate(
        outer_cv.split(np.zeros(len(labels)), labels, groups), start=1
    ):
        inner_labels = labels[train_val_idx]
        inner_groups = groups[train_val_idx]

        inner_cv = StratifiedGroupKFold(n_splits=n_inner_splits, shuffle=True, random_state=seed + outer_fold)
        inner_train_rel, inner_val_rel = next(
            inner_cv.split(np.zeros(len(train_val_idx)), inner_labels, inner_groups)
        )

        train_idx = np.array(train_val_idx)[inner_train_rel]
        val_idx = np.array(train_val_idx)[inner_val_rel]

        yield outer_fold, train_idx, val_idx, np.array(test_idx)

In [4]:
# ==============================
# Modelo
# ==============================

def create_model(model_name: str = MODEL_NAME) -> nn.Module:
    """
    Cria o backbone de transfer learning.

    A extração da penúltima camada será feita via hook no avgpool:
    - EfficientNet-B0 -> 1280 features
    - ResNet-50       -> 2048 features
    """
    model_name = model_name.lower()

    if model_name == "resnet50":
        model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        num_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(num_features, 1),
        )
    elif model_name == "efficientnet":
        model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        num_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(num_features, 1),
        )
    else:
        raise ValueError(f"MODEL_NAME inválido: {model_name}")

    return model.to(DEVICE)


def get_penultimate_hook_layer(model: nn.Module) -> nn.Module:
    """
    Retorna a camada onde o hook de features deve ser registrado.

    Usamos a saída do avgpool, que é justamente o embedding global imediatamente
    antes da cabeça classificadora.
    """
    if hasattr(model, "avgpool"):
        return model.avgpool

    raise AttributeError("Não foi possível localizar a camada de avgpool do modelo.")


def get_gradcam_target_layer(model: nn.Module) -> nn.Module:
    """
    Escolhe uma camada convolucional profunda para Grad-CAM.

    A regra abaixo funciona para os dois backbones usados.
    """
    if MODEL_NAME.lower() == "resnet50":
        return model.layer4[-1]
    if MODEL_NAME.lower() == "efficientnet":
        return model.features[-1]
    raise ValueError(f"Grad-CAM não implementado para {MODEL_NAME}")

In [5]:
# ==============================
# Métricas
# ==============================

def safe_roc_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, y_score))
    except Exception:
        return float("nan")


def safe_pr_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    try:
        return float(average_precision_score(y_true, y_score))
    except Exception:
        return float("nan")


def calculate_binary_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> Dict[str, object]:
    """
    Calcula métricas clássicas de classificação binária.
    """
    y_true = np.asarray(y_true).astype(int).ravel()
    y_score = np.asarray(y_score).astype(float).ravel()
    y_pred = (y_score >= threshold).astype(int)

    return {
        "auc": safe_roc_auc(y_true, y_score),
        "pr_auc": safe_pr_auc(y_true, y_score),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "n_samples": int(len(y_true)),
        "positive_rate": float(np.mean(y_true)),
    }


def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Tuple[float, Dict[str, object], np.ndarray, np.ndarray]:
    """
    Avalia a CNN em um DataLoader e retorna:
    - loss média
    - dicionário de métricas
    - y_true
    - y_score
    """
    model.eval()
    losses: List[float] = []
    all_scores: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []

    with torch.no_grad():
        for inputs, labels, _paths in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            logits = model(inputs)
            loss = criterion(logits, labels)

            scores = torch.sigmoid(logits).cpu().numpy().ravel()
            y_true = labels.cpu().numpy().ravel()

            losses.append(loss.item())
            all_scores.append(scores)
            all_labels.append(y_true)

    y_true_all = np.concatenate(all_labels) if all_labels else np.array([], dtype=int)
    y_score_all = np.concatenate(all_scores) if all_scores else np.array([], dtype=float)
    metrics = calculate_binary_metrics(y_true_all, y_score_all)

    return float(np.mean(losses) if losses else float("nan")), metrics, y_true_all, y_score_all

In [6]:

# ==============================
# Treinamento
# ==============================

@dataclass
class TrainingHistory:
    train_loss: List[float] = field(default_factory=list)
    val_loss: List[float] = field(default_factory=list)
    val_auc: List[float] = field(default_factory=list)
    val_f1: List[float] = field(default_factory=list)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    fold: int,
) -> Tuple[TrainingHistory, Path]:
    """
    Treina a CNN com early stopping baseado em val_loss.

    Salvamos o checkpoint selecionado do fold, que será usado depois na avaliação
    e na extração de features.
    """
    history = TrainingHistory()
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    checkpoint_path = DIR_MODELS / f"{TASK}_{MODEL_NAME}_fold{fold}_selected.pth"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        n_seen = 0

        for inputs, labels, _paths in train_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            batch_size = inputs.size(0)
            running_loss += loss.item() * batch_size
            n_seen += batch_size

        train_loss = running_loss / max(n_seen, 1)

        val_loss, val_metrics, _, _ = evaluate_model(model, val_loader, criterion)

        history.train_loss.append(float(train_loss))
        history.val_loss.append(float(val_loss))
        history.val_auc.append(float(val_metrics["auc"]))
        history.val_f1.append(float(val_metrics["f1"]))

        print(
            f"Fold {fold:02d} | Epoch {epoch:03d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_auc={val_metrics['auc']:.4f} | val_f1={val_metrics['f1']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping no fold {fold}: {PATIENCE} épocas sem melhora.")
            break

    return history, checkpoint_path

In [7]:
# ==============================
# Extração de features e PCA
# ==============================

def extract_features(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """
    Extrai o embedding da penúltima camada usando forward hook.

    Retorna:
    - X_features: ndarray (N, D)
    - y_labels: ndarray (N,)
    - paths: lista de caminhos das imagens
    """
    hook_layer = get_penultimate_hook_layer(model)
    feature_batches: List[torch.Tensor] = []
    label_batches: List[np.ndarray] = []
    path_list: List[str] = []

    def hook_fn(_module, _inputs, output):
        feature_batches.append(output.flatten(start_dim=1).detach().cpu())

    handle = hook_layer.register_forward_hook(hook_fn)

    model.eval()
    with torch.no_grad():
        for inputs, labels, paths in loader:
            inputs = inputs.to(DEVICE)
            _ = model(inputs)
            label_batches.append(np.asarray(labels))
            path_list.extend(list(paths))

    handle.remove()

    X_features = torch.cat(feature_batches, dim=0).numpy() if feature_batches else np.empty((0, 0))
    y_labels = np.concatenate(label_batches) if label_batches else np.array([], dtype=int)

    return X_features, y_labels, path_list


def fit_pca_on_train(
    X_train: np.ndarray,
    n_components: int = PCA_N_COMPONENTS,
) -> Tuple[StandardScaler, PCA, np.ndarray, np.ndarray, np.ndarray]:
    """
    Ajusta scaler + PCA apenas no treino.
    """
    effective_components = min(n_components, X_train.shape[0], X_train.shape[1])
    if effective_components < 2:
        raise ValueError(
            "O número efetivo de componentes PCA ficou < 2. "
            "Verifique o tamanho do conjunto de treino e a dimensionalidade das features."
        )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    pca = PCA(n_components=effective_components, random_state=SEED)
    X_train_pca = pca.fit_transform(X_train_scaled)

    explained = pca.explained_variance_ratio_
    cumulative = np.cumsum(explained)
    return scaler, pca, X_train_pca, explained, cumulative


def apply_pca(
    scaler: StandardScaler,
    pca: PCA,
    X: np.ndarray,
) -> np.ndarray:
    """
    Transforma um novo conjunto usando scaler e PCA já ajustados no treino.
    """
    X_scaled = scaler.transform(X)
    return pca.transform(X_scaled)

In [8]:

# ==============================
# OCPC
# ==============================

def normalize_binary_probabilities(prob_matrix: np.ndarray) -> np.ndarray:
    """
    Extrai a coluna da classe positiva.
    O `predict_proba` do OCPC retorna duas colunas em problema binário.
    """
    prob_matrix = np.asarray(prob_matrix, dtype=float)
    if prob_matrix.ndim == 1:
        return prob_matrix
    if prob_matrix.shape[1] == 1:
        return prob_matrix[:, 0]
    return prob_matrix[:, 1]


def _sanitize_for_ocpc(
    X: np.ndarray,
    y: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Garante tipos e layout de memória esperados pelo ocpc_py:
    - X: float64 C-contiguous 2-D
    - y: int32 1-D
    """
    X = np.asarray(X, dtype=np.float64)
    if not X.flags["C_CONTIGUOUS"]:
        X = np.ascontiguousarray(X)
    y = np.asarray(y, dtype=np.int32).ravel()
    return X, y


def fit_ocpc_and_predict(
    X_train_pca: np.ndarray,
    y_train: np.ndarray,
    X_eval_pca: np.ndarray,
) -> Tuple[object, np.ndarray, np.ndarray]:
    """
    Ajusta o OCPC no treino e produz:
    - y_score : probabilidade/score da classe positiva
    - y_pred  : classe predita

    Em caso de falha (ex: fold degenerado), retorna scores neutros
    e emite aviso sem interromper o pipeline.
    """
    X_tr, y_tr = _sanitize_for_ocpc(X_train_pca, y_train)
    X_ev = np.ascontiguousarray(np.asarray(X_eval_pca, dtype=np.float64))
    n_eval = X_ev.shape[0]

    try:
        clf = MultiClassPC()
        clf.fit(X_tr, y_tr)

        y_pred = np.asarray(clf.predict(X_ev)).astype(int).ravel()

        if hasattr(clf, "predict_proba"):
            y_score = normalize_binary_probabilities(clf.predict_proba(X_ev))
        else:
            y_score = y_pred.astype(float)

        return clf, y_score, y_pred

    except Exception as exc:
        warnings.warn(
            f"[OCPC] fit/predict falhou: {exc}\n"
            "Scores neutros (0.5) serão usados para este fold.",
            RuntimeWarning,
            stacklevel=2,
        )
        return None, np.full(n_eval, 0.5), np.zeros(n_eval, dtype=int)


def compute_geometric_metrics(
    X_pca: np.ndarray,
    y_true: np.ndarray,
    y_score: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, float]:
    """
    Métricas geométricas para descrever o espaço latente no paper.
    """
    metrics: Dict[str, float] = {}

    if X_pca.shape[1] >= 2 and len(np.unique(y_true)) > 1:
        X_2d = X_pca[:, :2]
        try:
            metrics["silhouette_pc1_pc2"] = float(silhouette_score(X_2d, y_true))
        except Exception:
            metrics["silhouette_pc1_pc2"] = float("nan")

        pos = X_2d[y_true == 1]
        neg = X_2d[y_true == 0]
        if len(pos) > 0 and len(neg) > 0:
            metrics["centroid_distance_pc1_pc2"] = float(
                np.linalg.norm(pos.mean(axis=0) - neg.mean(axis=0))
            )
        else:
            metrics["centroid_distance_pc1_pc2"] = float("nan")
    else:
        metrics["silhouette_pc1_pc2"] = float("nan")
        metrics["centroid_distance_pc1_pc2"] = float("nan")

    metrics["ocpc_auc"] = safe_roc_auc(y_true, y_score)
    metrics["ocpc_pr_auc"] = safe_pr_auc(y_true, y_score)
    metrics["ocpc_f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    metrics["ocpc_accuracy"] = float(accuracy_score(y_true, y_pred))
    return metrics


In [9]:
# ==============================
# Grad-CAM
# ==============================

def denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
    """
    Converte tensor normalizado (C,H,W) para ndarray RGB em [0,1].
    """
    image = image_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    image = (image * IMAGENET_STD) + IMAGENET_MEAN
    image = np.clip(image, 0.0, 1.0)
    return image


def generate_gradcam_overlay(model: nn.Module, image_tensor: torch.Tensor) -> np.ndarray:
    """
    Gera overlay do Grad-CAM.
    """
    if GradCAM is None or show_cam_on_image is None:
        raise ImportError("pytorch-grad-cam não está disponível no ambiente.")

    target_layer = get_gradcam_target_layer(model)
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0).to(DEVICE))[0]
    rgb_image = denormalize_image(image_tensor)
    overlay = show_cam_on_image(rgb_image, grayscale_cam, use_rgb=True)
    return overlay


def save_gradcam_examples(
    model: nn.Module,
    dataset: Dataset,
    fold: int,
    max_images: int = MAX_GRADCAM_IMAGES_PER_FOLD,
) -> None:
    """
    Salva algumas imagens Grad-CAM do conjunto de teste.

    Estratégia:
    - percorremos o loader em batch 1
    - coletamos exemplos corretamente classificados de ambas as classes
    - limitamos o número de figuras para não poluir o experimento
    """
    if not SAVE_GRADCAM_EXAMPLES:
        return
    if GradCAM is None or show_cam_on_image is None:
        print("Grad-CAM indisponível no ambiente; pulando visualização.")
        return

    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
    saved = 0
    found_per_class = {0: 0, 1: 0}

    model.eval()
    with torch.no_grad():
        for image, label, path in loader:
            image = image.to(DEVICE)
            logits = model(image)
            score = torch.sigmoid(logits).cpu().numpy().ravel()[0]
            pred = int(score >= 0.5)
            true = int(label.numpy().ravel()[0])

            if pred != true:
                continue
            if found_per_class[true] >= max(1, max_images // 2):
                continue

            overlay = generate_gradcam_overlay(model, image.squeeze(0).cpu())

            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(denormalize_image(image.squeeze(0).cpu()))
            axes[0].set_title(f"Original\nTrue={true} | Pred={pred} | p={score:.3f}")
            axes[0].axis("off")

            axes[1].imshow(overlay)
            axes[1].set_title("Grad-CAM")
            axes[1].axis("off")

            fname = Path(path[0]).stem
            out_path = DIR_GRADCAM / f"fold{fold}_{fname}_gradcam.png"
            plt.tight_layout()
            plt.savefig(out_path, dpi=180, bbox_inches="tight")
            plt.close(fig)

            found_per_class[true] += 1
            saved += 1

            if saved >= max_images:
                break

In [10]:
# ==============================
# Relatórios e visualizações
# ==============================

@dataclass
class FoldArtifact:
    fold: int
    train_history: TrainingHistory
    train_metrics_cnn: Dict[str, object]
    val_metrics_cnn: Dict[str, object]
    test_metrics_cnn: Dict[str, object]
    val_metrics_ocpc: Dict[str, object]
    test_metrics_ocpc: Dict[str, object]
    geometric_metrics_test: Dict[str, float]
    explained_variance_ratio: np.ndarray
    cumulative_variance: np.ndarray
    pca_train_2d: np.ndarray
    pca_val_2d: np.ndarray
    pca_test_2d: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    ocpc_test_scores: np.ndarray
    cnn_test_scores: np.ndarray
    emissions_kg: float


class ReportManager:
    """
    Organiza resultados por fold e gera tabelas/figuras consolidadas.

    As visualizações foram pensadas para ajudar tanto na análise do experimento
    quanto na redação do paper.
    """

    def __init__(self):
        self.artifacts: List[FoldArtifact] = []

    def add(self, artifact: FoldArtifact) -> None:
        self.artifacts.append(artifact)

    def finalize(self) -> None:
        self._export_metrics_csv()
        self._plot_training_history()
        self._plot_metric_panels()
        self._plot_confusion_matrices()
        self._plot_roc_pr_curves()
        self._plot_pca_scatter_by_fold()
        self._plot_pca_variance()
        self._plot_ocpc_score_distributions()
        self._plot_emissions()

    def _export_metrics_csv(self) -> None:
        rows = []
        for art in self.artifacts:
            row = {"fold": art.fold, "emissions_kg_co2eq": art.emissions_kg}

            for prefix, metrics in [
                ("cnn_train", art.train_metrics_cnn),
                ("cnn_val", art.val_metrics_cnn),
                ("cnn_test", art.test_metrics_cnn),
                ("ocpc_val", art.val_metrics_ocpc),
                ("ocpc_test", art.test_metrics_ocpc),
                ("geo_test", art.geometric_metrics_test),
            ]:
                for key, value in metrics.items():
                    if key == "confusion_matrix":
                        continue
                    row[f"{prefix}_{key}"] = value

            rows.append(row)

        df = pd.DataFrame(rows)
        summary = {"fold": "mean±std"}
        for col in df.columns:
            if col == "fold":
                continue
            if pd.api.types.is_numeric_dtype(df[col]):
                summary[col] = f"{df[col].mean():.4f} ± {df[col].std():.4f}"

        out_df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
        out_df.to_csv(DIR_METRICS / f"consolidated_{TASK}_{MODEL_NAME}.csv", index=False)

        df.to_excel(DIR_METRICS / f"consolidated_{TASK}_{MODEL_NAME}.xlsx", index=False)

    def _plot_training_history(self) -> None:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        fig.suptitle(f"Histórico de treinamento por fold — {TASK} ({MODEL_NAME})", fontsize=13)

        for art in self.artifacts:
            color = FOLD_COLORS[(art.fold - 1) % len(FOLD_COLORS)]
            epochs = np.arange(1, len(art.train_history.train_loss) + 1)

            axes[0].plot(epochs, art.train_history.train_loss, "--", color=color, alpha=0.8, label=f"Fold {art.fold} train")
            axes[0].plot(epochs, art.train_history.val_loss, "-", color=color, linewidth=2, label=f"Fold {art.fold} val")
            axes[1].plot(epochs, art.train_history.val_auc, color=color, linewidth=2, label=f"Fold {art.fold}")
            axes[2].plot(epochs, art.train_history.val_f1, color=color, linewidth=2, label=f"Fold {art.fold}")

        titles = ["Loss", "AUC-ROC (val)", "F1-score (val)"]
        ylabels = ["Loss", "AUC", "F1"]
        for ax, title, ylabel in zip(axes, titles, ylabels):
            ax.set_title(title)
            ax.set_xlabel("Época")
            ax.set_ylabel(ylabel)
            ax.grid(alpha=0.3)
            ax.legend(fontsize=7, ncol=2)

        plt.tight_layout()
        plt.savefig(DIR_FIGURES / f"training_history_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _plot_metric_panels(self) -> None:
        metrics_to_show = ["auc", "pr_auc", "f1", "accuracy", "precision", "recall"]
        pretty_names = {
            "auc": "AUC-ROC",
            "pr_auc": "PR-AUC",
            "f1": "F1-score",
            "accuracy": "Accuracy",
            "precision": "Precision",
            "recall": "Recall",
        }

        for family, title_prefix in [("cnn", "CNN"), ("ocpc", "OCPC")]:
            fig, axes = plt.subplots(2, 3, figsize=(15, 8))
            fig.suptitle(f"{title_prefix} — métricas por fold ({TASK}, {MODEL_NAME})", fontsize=13)

            folds = [art.fold for art in self.artifacts]
            x = np.arange(len(folds))

            for ax, metric_name in zip(axes.ravel(), metrics_to_show):
                val_values = [getattr(art, f"val_metrics_{family}")[metric_name] for art in self.artifacts]
                test_values = [getattr(art, f"test_metrics_{family}")[metric_name] for art in self.artifacts]

                bars1 = ax.bar(x - 0.18, val_values, width=0.36, label="Val", color="#4C72B0", alpha=0.8)
                bars2 = ax.bar(x + 0.18, test_values, width=0.36, label="Test", color="#DD8452", alpha=0.8)

                ax.set_xticks(x)
                ax.set_xticklabels([f"F{f}" for f in folds])
                ax.set_ylim(0.0, 1.08)
                ax.set_title(pretty_names[metric_name])
                ax.grid(axis="y", alpha=0.3)
                ax.legend(fontsize=8)

                for bar in list(bars1) + list(bars2):
                    value = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.3f}",
                            ha="center", va="bottom", fontsize=7)

            plt.tight_layout()
            plt.savefig(DIR_FIGURES / f"{family}_metric_panel_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
            plt.close(fig)

    def _plot_confusion_matrices(self) -> None:
        for family, title_prefix in [("cnn", "CNN"), ("ocpc", "OCPC")]:
            for split in ["val", "test"]:
                n = len(self.artifacts)
                fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
                if n == 1:
                    axes = [axes]

                fig.suptitle(f"{title_prefix} — confusion matrices ({split})", fontsize=12)

                for ax, art in zip(axes, self.artifacts):
                    cm = getattr(art, f"{split}_metrics_{family}")["confusion_matrix"].astype(float)
                    row_sums = cm.sum(axis=1, keepdims=True)
                    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)

                    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
                    for r in range(cm.shape[0]):
                        for c in range(cm.shape[1]):
                            color = "white" if cm_norm[r, c] > 0.55 else "black"
                            ax.text(c, r, f"{int(cm[r, c])}\n({cm_norm[r, c]:.0%})",
                                    ha="center", va="center", color=color, fontsize=9)

                    ax.set_title(f"Fold {art.fold}")
                    ax.set_xticks([0, 1])
                    ax.set_yticks([0, 1])
                    ax.set_xticklabels(["Pred 0", "Pred 1"])
                    ax.set_yticklabels(["True 0", "True 1"])

                plt.tight_layout()
                plt.savefig(DIR_FIGURES / f"{family}_{split}_confusion_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
                plt.close(fig)

    def _plot_roc_pr_curves(self) -> None:
        """
        Curvas ROC/PR do conjunto de teste por fold.

        Essas figuras costumam ser muito úteis em artigos quando a base é
        potencialmente desbalanceada.
        """
        for family, title_prefix in [("cnn", "CNN"), ("ocpc", "OCPC")]:
            fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
            fig_pr, ax_pr = plt.subplots(figsize=(6, 5))

            for art in self.artifacts:
                y_true = art.y_test
                y_score = art.cnn_test_scores if family == "cnn" else art.ocpc_test_scores

                if len(np.unique(y_true)) < 2:
                    continue

                fpr, tpr, _ = roc_curve(y_true, y_score)
                precision, recall, _ = precision_recall_curve(y_true, y_score)

                roc_auc = safe_roc_auc(y_true, y_score)
                pr_auc = safe_pr_auc(y_true, y_score)
                color = FOLD_COLORS[(art.fold - 1) % len(FOLD_COLORS)]

                ax_roc.plot(fpr, tpr, color=color, linewidth=2, label=f"Fold {art.fold} (AUC={roc_auc:.3f})")
                ax_pr.plot(recall, precision, color=color, linewidth=2, label=f"Fold {art.fold} (AP={pr_auc:.3f})")

            ax_roc.plot([0, 1], [0, 1], "--", color="gray", alpha=0.7)
            ax_roc.set_title(f"{title_prefix} — ROC no teste")
            ax_roc.set_xlabel("FPR")
            ax_roc.set_ylabel("TPR")
            ax_roc.grid(alpha=0.3)
            ax_roc.legend(fontsize=8)

            ax_pr.set_title(f"{title_prefix} — Precision-Recall no teste")
            ax_pr.set_xlabel("Recall")
            ax_pr.set_ylabel("Precision")
            ax_pr.grid(alpha=0.3)
            ax_pr.legend(fontsize=8)

            fig_roc.tight_layout()
            fig_pr.tight_layout()

            fig_roc.savefig(DIR_FIGURES / f"{family}_roc_test_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
            fig_pr.savefig(DIR_FIGURES / f"{family}_pr_test_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
            plt.close(fig_roc)
            plt.close(fig_pr)

    def _plot_pca_scatter_by_fold(self) -> None:
        """
        Scatter de PC1 vs PC2 por fold, destacando train/val/test.

        Essa figura é útil no paper para mostrar:
        - distribuição das classes no espaço latente
        - diferença visual entre treino, validação e teste
        - coerência do embedding aprendido
        """
        n = len(self.artifacts)
        cols = min(3, n)
        rows = int(math.ceil(n / cols))

        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
        axes = np.array(axes).reshape(-1)

        for ax, art in zip(axes, self.artifacts):
            self._scatter_split(ax, art.pca_train_2d, art.y_train, "train")
            self._scatter_split(ax, art.pca_val_2d, art.y_val, "val")
            self._scatter_split(ax, art.pca_test_2d, art.y_test, "test")

            ax.set_title(f"Fold {art.fold}")
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            ax.grid(alpha=0.2)

        for ax in axes[len(self.artifacts):]:
            ax.set_visible(False)

        handles = [
            plt.Line2D([], [], linestyle="", marker="o", color=SPLIT_COLORS["train"], label="Train"),
            plt.Line2D([], [], linestyle="", marker="o", color=SPLIT_COLORS["val"], label="Val"),
            plt.Line2D([], [], linestyle="", marker="o", color=SPLIT_COLORS["test"], label="Test"),
        ]
        fig.legend(handles=handles, loc="upper center", ncol=3)
        fig.suptitle(f"PCA (PC1 × PC2) por fold — {TASK} ({MODEL_NAME})", fontsize=13)
        plt.tight_layout()
        plt.savefig(DIR_PCA_FIG / f"pca_split_scatter_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _scatter_split(self, ax, X_2d: np.ndarray, y: np.ndarray, split_name: str) -> None:
        color = SPLIT_COLORS[split_name]
        markers = {0: "o", 1: "^"}

        for cls in [0, 1]:
            mask = (y == cls)
            if np.sum(mask) == 0:
                continue
            ax.scatter(
                X_2d[mask, 0],
                X_2d[mask, 1],
                s=22,
                alpha=0.60 if split_name != "train" else 0.35,
                c=color,
                marker=markers[cls],
                edgecolors="none",
            )

    def _plot_pca_variance(self) -> None:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        for art in self.artifacts:
            cumulative_pct = art.cumulative_variance * 100
            ax.plot(
                np.arange(1, len(cumulative_pct) + 1),
                cumulative_pct,
                linewidth=2,
                color=FOLD_COLORS[(art.fold - 1) % len(FOLD_COLORS)],
                label=f"Fold {art.fold}",
            )

        ax.axhline(90, linestyle="--", color="gray", alpha=0.7)
        ax.axhline(95, linestyle=":", color="gray", alpha=0.7)
        ax.set_xlabel("Número de componentes")
        ax.set_ylabel("Variância acumulada (%)")
        ax.set_title(f"Scree plot acumulado — {TASK} ({MODEL_NAME})")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(DIR_PCA_FIG / f"scree_cumulative_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _plot_ocpc_score_distributions(self) -> None:
        """
        Boxplots dos scores do OCPC no teste, separados por classe real.
        """
        rows = []
        for art in self.artifacts:
            for score, label in zip(art.ocpc_test_scores, art.y_test):
                rows.append({
                    "fold": art.fold,
                    "score": float(score),
                    "label": int(label),
                })

        df = pd.DataFrame(rows)
        if df.empty:
            return

        fig, ax = plt.subplots(figsize=(9, 5))
        positions = []
        labels = []
        data = []

        current_pos = 1
        for fold in sorted(df["fold"].unique()):
            for cls in [0, 1]:
                subset = df[(df["fold"] == fold) & (df["label"] == cls)]["score"].values
                if len(subset) == 0:
                    continue
                positions.append(current_pos)
                labels.append(f"F{fold}\nC{cls}")
                data.append(subset)
                current_pos += 1
            current_pos += 0.5

        bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
        for patch, lab in zip(bp["boxes"], labels):
            cls = int(lab.split("\n")[1].replace("C", ""))
            patch.set_facecolor(CLASS_COLORS[cls])
            patch.set_alpha(0.6)

        ax.set_xticks(positions)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Score OCPC para classe positiva")
        ax.set_title(f"Distribuição dos scores do OCPC no teste — {TASK} ({MODEL_NAME})")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(DIR_FIGURES / f"ocpc_score_distribution_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _plot_emissions(self) -> None:
        emissions = [art.emissions_kg for art in self.artifacts]
        if len(emissions) == 0:
            return

        folds = [art.fold for art in self.artifacts]
        cumulative = np.cumsum(emissions)
        total = float(np.sum(emissions))

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        axes[0].bar([f"Fold {f}" for f in folds], emissions, color=FOLD_COLORS[:len(folds)], alpha=0.85)
        axes[0].set_title("Emissões por fold")
        axes[0].set_ylabel("kg CO₂eq")
        axes[0].grid(axis="y", alpha=0.3)

        axes[1].plot(range(1, len(cumulative) + 1), cumulative, marker="o", linewidth=2)
        axes[1].fill_between(range(1, len(cumulative) + 1), cumulative, alpha=0.15)
        axes[1].set_xticks(range(1, len(cumulative) + 1))
        axes[1].set_xticklabels([f"Fold {f}" for f in folds])
        axes[1].set_title(f"Acumulado (total = {total * 1000:.1f} g CO₂eq)")
        axes[1].set_ylabel("kg CO₂eq")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(DIR_FIGURES / f"emissions_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

        pd.DataFrame({"fold": folds, "kg_co2eq": emissions, "kg_co2eq_cumulative": cumulative}).to_csv(
            DIR_METRICS / f"emissions_{TASK}_{MODEL_NAME}.csv",
            index=False
        )

In [11]:
# Carbon tracking
# ==============================

class NullTracker:
    """Fallback simples quando CodeCarbon não estiver disponível."""
    def start(self):
        return None

    def stop(self):
        return 0.0


def create_emissions_tracker(fold: int):
    """
    Cria um tracker de emissões de carbono.

    Se o CodeCarbon não estiver instalado, o experimento continua normalmente.
    """
    if not USE_CODECARBON or codecarbon is None:
        return NullTracker()

    try:
        return codecarbon.EmissionsTracker(
            output_dir=str(DIR_EMISSIONS),
            output_file=f"fold{fold:02d}_train_emissions.csv",
            log_level="error",
        )
    except Exception:
        return NullTracker()


In [12]:
#==============================
# Pipeline principal
# ==============================

def run_cross_validation() -> None:
    set_seed(SEED)

    dataset_for_split = DFUDataset(DATA_ROOT, transform=eval_transform)
    labels = np.asarray(dataset_for_split.labels)
    groups = np.asarray(dataset_for_split.identifiers)

    report = ReportManager()

    split_rows = []

    for fold, train_idx, val_idx, test_idx in get_nested_group_split(labels, groups):
        print("\n" + "=" * 80)
        print(f"FOLD {fold}/{N_FOLDS}")
        print("=" * 80)

        # Registramos os caminhos de cada split para auditoria do paper.
        split_rows.extend([
            {"fold": fold, "split": "train", "path": str(dataset_for_split.image_paths[i]), "label": int(labels[i]), "group": groups[i]}
            for i in train_idx
        ])
        split_rows.extend([
            {"fold": fold, "split": "val", "path": str(dataset_for_split.image_paths[i]), "label": int(labels[i]), "group": groups[i]}
            for i in val_idx
        ])
        split_rows.extend([
            {"fold": fold, "split": "test", "path": str(dataset_for_split.image_paths[i]), "label": int(labels[i]), "group": groups[i]}
            for i in test_idx
        ])

        # Subsets com augmentation no treino e transformação determinística em val/test.
        train_dataset = build_subset(dataset_for_split, train_idx, train_transform)
        val_dataset = build_subset(dataset_for_split, val_idx, eval_transform)
        test_dataset = build_subset(dataset_for_split, test_idx, eval_transform)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        model = create_model(MODEL_NAME)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

        tracker = create_emissions_tracker(fold)
        tracker.start()
        history, checkpoint_path = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            fold=fold,
        )
        emissions_kg = float(tracker.stop() or 0.0)

        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=False))
        model.eval()

        # Avaliação da CNN
        train_eval_loader = DataLoader(build_subset(dataset_for_split, train_idx, eval_transform),
                                       batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        _, train_metrics_cnn, y_train_eval, y_train_score_cnn = evaluate_model(model, train_eval_loader, criterion)
        _, val_metrics_cnn, y_val, y_val_score_cnn = evaluate_model(model, val_loader, criterion)
        _, test_metrics_cnn, y_test, y_test_score_cnn = evaluate_model(model, test_loader, criterion)

        # Extração de features separada por split para evitar leakage.
        X_train_feat, y_train, train_paths = extract_features(model, train_eval_loader)
        X_val_feat, y_val_feat, val_paths = extract_features(model, val_loader)
        X_test_feat, y_test_feat, test_paths = extract_features(model, test_loader)

        # Sanidade: labels da extração devem coincidir com os labels da avaliação.
        assert np.array_equal(y_train, y_train_eval.astype(int))
        assert np.array_equal(y_val_feat, y_val.astype(int))
        assert np.array_equal(y_test_feat, y_test.astype(int))

        scaler, pca, X_train_pca, explained, cumulative = fit_pca_on_train(X_train_feat, PCA_N_COMPONENTS)
        X_val_pca = apply_pca(scaler, pca, X_val_feat)
        X_test_pca = apply_pca(scaler, pca, X_test_feat)

        # OCPC no espaço PCA.
        ocpc_model, y_val_score_ocpc, y_val_pred_ocpc = fit_ocpc_and_predict(X_train_pca, y_train, X_val_pca)
        _, y_test_score_ocpc, y_test_pred_ocpc = fit_ocpc_and_predict(X_train_pca, y_train, X_test_pca)

        val_metrics_ocpc = calculate_binary_metrics(y_val, y_val_score_ocpc)
        test_metrics_ocpc = calculate_binary_metrics(y_test, y_test_score_ocpc)
        geometric_metrics_test = compute_geometric_metrics(X_test_pca, y_test, y_test_score_ocpc, y_test_pred_ocpc)

        # Salva features PCA do teste e do treino para análises externas.
        train_pca_df = pd.DataFrame(X_train_pca[:, :min(10, X_train_pca.shape[1])],
                                    columns=[f"PC{i+1}" for i in range(min(10, X_train_pca.shape[1]))])
        train_pca_df["label"] = y_train
        train_pca_df["split"] = "train"
        train_pca_df["fold"] = fold
        train_pca_df.to_csv(DIR_METRICS / f"pca_train_fold{fold}.csv", index=False)

        test_pca_df = pd.DataFrame(X_test_pca[:, :min(10, X_test_pca.shape[1])],
                                   columns=[f"PC{i+1}" for i in range(min(10, X_test_pca.shape[1]))])
        test_pca_df["label"] = y_test
        test_pca_df["split"] = "test"
        test_pca_df["fold"] = fold
        test_pca_df.to_csv(DIR_METRICS / f"pca_test_fold{fold}.csv", index=False)

        # Grad-CAM opcional, útil para o paper.
        if SAVE_GRADCAM_EXAMPLES:
            try:
                save_gradcam_examples(model, test_dataset, fold=fold)
            except Exception as exc:
                print(f"Grad-CAM do fold {fold} não pôde ser salvo: {exc}")

        artifact = FoldArtifact(
            fold=fold,
            train_history=history,
            train_metrics_cnn=train_metrics_cnn,
            val_metrics_cnn=val_metrics_cnn,
            test_metrics_cnn=test_metrics_cnn,
            val_metrics_ocpc=val_metrics_ocpc,
            test_metrics_ocpc=test_metrics_ocpc,
            geometric_metrics_test=geometric_metrics_test,
            explained_variance_ratio=explained,
            cumulative_variance=cumulative,
            pca_train_2d=X_train_pca[:, :2],
            pca_val_2d=X_val_pca[:, :2],
            pca_test_2d=X_test_pca[:, :2],
            y_train=y_train,
            y_val=y_val.astype(int),
            y_test=y_test.astype(int),
            ocpc_test_scores=y_test_score_ocpc,
            cnn_test_scores=y_test_score_cnn,
            emissions_kg=emissions_kg,
        )
        report.add(artifact)

        print(
            f"[Fold {fold}] CNN test AUC={test_metrics_cnn['auc']:.4f} | "
            f"OCPC test AUC={test_metrics_ocpc['auc']:.4f} | "
            f"Silhouette={geometric_metrics_test['silhouette_pc1_pc2']:.4f} | "
            f"CO2={emissions_kg * 1000:.2f} g"
        )

    pd.DataFrame(split_rows).to_csv(DIR_METRICS / f"split_audit_{TASK}_{MODEL_NAME}.csv", index=False)
    report.finalize()
    print("Pipeline concluído com sucesso.")


In [13]:
# Ponto de entrada
# ═══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    run_cross_validation()


FOLD 1/5


[codecarbon WARNING @ 16:11:21] Multiple instances of codecarbon are allowed to run at the same time.


Fold 01 | Epoch 001/100 | train_loss=0.6264 | val_loss=0.5398 | val_auc=0.9065 | val_f1=0.8325
Fold 01 | Epoch 002/100 | train_loss=0.4700 | val_loss=0.3815 | val_auc=0.9566 | val_f1=0.8815
Fold 01 | Epoch 003/100 | train_loss=0.3409 | val_loss=0.2744 | val_auc=0.9755 | val_f1=0.9298
Fold 01 | Epoch 004/100 | train_loss=0.2669 | val_loss=0.2229 | val_auc=0.9794 | val_f1=0.9378
Fold 01 | Epoch 005/100 | train_loss=0.2220 | val_loss=0.1942 | val_auc=0.9791 | val_f1=0.9345
Fold 01 | Epoch 006/100 | train_loss=0.1883 | val_loss=0.1863 | val_auc=0.9818 | val_f1=0.9396
Fold 01 | Epoch 007/100 | train_loss=0.1649 | val_loss=0.1765 | val_auc=0.9813 | val_f1=0.9335
Fold 01 | Epoch 008/100 | train_loss=0.1482 | val_loss=0.1783 | val_auc=0.9795 | val_f1=0.9331
Fold 01 | Epoch 009/100 | train_loss=0.1244 | val_loss=0.1738 | val_auc=0.9820 | val_f1=0.9318
Fold 01 | Epoch 010/100 | train_loss=0.1076 | val_loss=0.1743 | val_auc=0.9813 | val_f1=0.9282
Fold 01 | Epoch 011/100 | train_loss=0.1033 | val_

ValueError: setting an array element with a sequence.